# Isolated beta-plume example

This notebook computes a mechanically forced analogue of the beta plume introduced by Stommel (1982), *Is the South Pacific helium-3 plume dynamically active?* ([doi:10.1016/0012-821X(82)90038-3](https://doi.org/10.1016/0012-821X(82)90038-3)), with vorticity-budget context from Haine and Fuller (2016), *Boundary $\beta$-plumes and their vorticity budgets* ([doi:10.1002/qj.2866](https://doi.org/10.1002/qj.2866)). A smooth, isolated patch of positive wind-stress curl is placed east of the basin center. The bottom is flat and buoyancy forcing is identically zero, so the response isolates planetary-vorticity advection, friction, and the westward Sverdrup plume.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", "PlanetaryGeostrophy"))
using PlanetaryGeostrophy
using Plots

## Flat-bottom beta plane with isolated forcing

The azimuthal surface stress is constructed so its vertical curl is exactly the Gaussian

$$\hat{\boldsymbol z}\mathbin{\boldsymbol\cdot}(\boldsymbol\nabla\mathbin{\boldsymbol\times}\boldsymbol\tau)=C_0\exp[-r^2/(2\sigma^2)].$$

The forcing center lies east of the basin midpoint, leaving room for the plume to extend westward. The Coriolis parameter is $f=f_0+\beta y$.

In [ ]:
Lx, Ly = 4.0, 2.0
H0 = 1.0
f0, beta = 1.0, 0.30
Ek, Γ = 0.003, 0.20

forcing_center = (0.80, 0.0)
forcing_radius = 0.20
curl_amplitude = 8e-4

forcing_radius_squared(x, y) =
    (x - forcing_center[1])^2 + (y - forcing_center[2])^2

function stress_factor(x, y)
    r2 = forcing_radius_squared(x, y)
    r2 < 1e-14 && return curl_amplitude / 2
    return curl_amplitude * forcing_radius^2 *
        (-expm1(-r2 / (2forcing_radius^2))) / r2
end

tau_x(x, y) = -stress_factor(x, y) * (y - forcing_center[2])
tau_y(x, y) = stress_factor(x, y) * (x - forcing_center[1])
wind_curl(x, y) = curl_amplitude *
    exp(-forcing_radius_squared(x, y) / (2forcing_radius^2))

problem = PGProblem(
    parameters=PGParameters(Ek, Γ; viscosity=1.0, diffusivity=1.0),
    depth=(x, y) -> H0,
    depth_x=(x, y) -> 0.0,
    depth_y=(x, y) -> 0.0,
    coriolis=(x, y) -> f0 + beta * y,
    coriolis_x=(x, y) -> 0.0,
    coriolis_y=(x, y) -> beta,
    wind_stress=(x, y) -> tau_x(x, y) + im * tau_y(x, y),
    wind_stress_curl=wind_curl,
    restoring_buoyancy=(x, y, z) -> 0.0,
    surface_buoyancy=(x, y) -> 0.0,
    boundary=rectangle_boundary(Lx, Ly),
)

## Isolated forcing patch

In [ ]:
forcing_x = range(-Lx / 2, Lx / 2; length=161)
forcing_y = range(-Ly / 2, Ly / 2; length=81)
curl_field = [wind_curl(xx, yy) for xx in forcing_x, yy in forcing_y]
stress_magnitude = [
    hypot(tau_x(xx, yy), tau_y(xx, yy))
    for xx in forcing_x, yy in forcing_y
]

curl_panel = heatmap(
    forcing_x, forcing_y, curl_field';
    title="isolated curl(τ)", xlabel="x", ylabel="y",
    color=:thermal, aspect_ratio=:equal,
)
stress_panel = heatmap(
    forcing_x, forcing_y, stress_magnitude';
    title="surface-stress magnitude", xlabel="x", ylabel="y",
    color=:viridis, aspect_ratio=:equal,
)
forcing_figure = plot(
    curl_panel, stress_panel;
    layout=(1, 2), size=(1000, 400), plot_title="beta-plume forcing",
)
savefig(forcing_figure, joinpath(@__DIR__, "beta_plume_forcing.png"))
display(forcing_figure)

## Surface-pressure and transport solution

In [ ]:
solution = solve_pg(
    problem;
    mesh_file=joinpath(@__DIR__, "beta_plume_pressure.msh"),
    mesh_resolution=0.08,
    pressure_order=2,
    overwrite_mesh=true,
)
display(performance_summary(solution))

In [ ]:
# Stay just inside the finite-element boundary for point evaluation.
x = range(-0.99Lx / 2, 0.99Lx / 2; length=81)
y = range(-0.99Ly / 2, 0.99Ly / 2; length=41)
diagnostics = compute_diagnostics(solution, x, y)
display(performance_summary(diagnostics))

## Westward beta plume

The localized forcing drives meridional Sverdrup transport within the patch. Integrating that transport from the eastern boundary produces a streamfunction wake extending westward from the source. Bottom stress closes the circulation near the basin boundaries.

In [ ]:
transport_figure = plot_depth_integrated_current(
    diagnostics;
    arrow_scale=0.18,
    output=joinpath(@__DIR__, "beta_plume_transport.png"),
)
streamfunction_figure = plot_streamfunction(
    diagnostics;
    output=joinpath(@__DIR__, "beta_plume_streamfunction.png"),
)
bottom_stress_figure = plot_bottom_stress(
    diagnostics;
    arrow_scale=0.18,
    output=joinpath(@__DIR__, "beta_plume_bottom_stress.png"),
)
display(transport_figure)
display(streamfunction_figure)
display(bottom_stress_figure)

## Barotropic-vorticity closure

With flat depth and zero buoyancy, the interior balance is $\beta V=\mathrm{Ek}\,\operatorname{curl}(\tau)$. The bottom-stress curl supplies the departure from Sverdrup balance required to close the plume against the boundaries.

In [ ]:
barotropic_vorticity = compute_barotropic_vorticity_diagnostics(
    solution, diagnostics,
)
vorticity = compute_vorticity_diagnostics(solution, diagnostics)
display(performance_summary(barotropic_vorticity))
display(performance_summary(vorticity))

barotropic_vorticity_figure = plot_barotropic_vorticity_balance(
    barotropic_vorticity;
    output=joinpath(@__DIR__, "beta_plume_barotropic_vorticity.png"),
)
vorticity_figure = plot_vorticity_balance(
    vorticity;
    output=joinpath(@__DIR__, "beta_plume_vorticity.png"),
)
display(barotropic_vorticity_figure)
display(vorticity_figure)

## Sverdrup-plume comparison

In [ ]:
sverdrup_V = [Ek * wind_curl(xx, yy) / beta for xx in x, yy in y]
sverdrup_streamfunction = zeros(length(x), length(y))
for j in eachindex(y), i in (length(x) - 1):-1:1
    dx = x[i + 1] - x[i]
    sverdrup_streamfunction[i, j] =
        sverdrup_streamfunction[i + 1, j] -
        dx * (sverdrup_V[i + 1, j] + sverdrup_V[i, j]) / 2
end

planetary_vorticity = beta .* diagnostics.analytical_transport_V
surface_stress_curl = [Ek * wind_curl(xx, yy) for xx in x, yy in y]
sverdrup_departure = planetary_vorticity .- surface_stress_curl
sverdrup_departure[.!barotropic_vorticity.wet] .= NaN
interior_scale = maximum(abs, surface_stress_curl)
closure_scale = maximum(abs, [
    sverdrup_departure[barotropic_vorticity.wet];
    barotropic_vorticity.bottom_stress_curl[barotropic_vorticity.wet];
])

p1 = heatmap(x, y, surface_stress_curl';
    title="isolated Ek curl(τ)", xlabel="x", ylabel="y",
    color=:thermal, aspect_ratio=:equal)
p2 = contourf(x, y, sverdrup_streamfunction';
    title="Sverdrup beta plume", xlabel="x", ylabel="y",
    aspect_ratio=:equal)
p3 = contourf(x, y, diagnostics.streamfunction';
    title="PG streamfunction", xlabel="x", ylabel="y",
    aspect_ratio=:equal)
p4 = heatmap(x, y, planetary_vorticity';
    title="βV", xlabel="x", ylabel="y", color=:balance,
    clims=(-interior_scale, interior_scale), aspect_ratio=:equal)
p5 = heatmap(x, y, sverdrup_departure';
    title="βV - Ek curl(τ)", xlabel="x", ylabel="y", color=:balance,
    clims=(-closure_scale, closure_scale), aspect_ratio=:equal)
p6 = heatmap(x, y, barotropic_vorticity.bottom_stress_curl';
    title="Ek curl(τ_b)", xlabel="x", ylabel="y", color=:balance,
    clims=(-closure_scale, closure_scale), aspect_ratio=:equal)

comparison_figure = plot(
    p1, p2, p3, p4, p5, p6;
    layout=(2, 3), size=(1300, 760),
    plot_title="isolated forcing and westward beta plume",
)
savefig(comparison_figure, joinpath(@__DIR__, "beta_plume_comparison.png"))
display(comparison_figure)

## Plume diagnostics

The shape correlation compares the PG and inviscid Sverdrup streamfunctions west of the forcing while excluding the immediate western boundary current.

In [ ]:
westward_plume = [
    xx > -0.8Lx / 2 && xx < forcing_center[1] - 2forcing_radius &&
    abs(yy - forcing_center[2]) < 2forcing_radius
    for xx in x, yy in y
]
eastern_region = [
    xx > forcing_center[1] + 2forcing_radius &&
    abs(yy - forcing_center[2]) < 2forcing_radius
    for xx in x, yy in y
]

westward_peak = maximum(abs, diagnostics.streamfunction[westward_plume])
eastern_peak = maximum(abs, diagnostics.streamfunction[eastern_region])
westward_plume_ratio = westward_peak / eastern_peak
sverdrup_shape_correlation = (
    sum(
        diagnostics.streamfunction[westward_plume] .*
        sverdrup_streamfunction[westward_plume]
    ) /
    sqrt(
        sum(abs2, diagnostics.streamfunction[westward_plume]) *
        sum(abs2, sverdrup_streamfunction[westward_plume])
    )
)

display((;
    maximum_forcing_curl=maximum(curl_field),
    westward_peak,
    eastern_peak,
    westward_plume_ratio,
    sverdrup_shape_correlation,
))